# Inference (Single User)

Run recommendations for one user using the saved BPR model.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Workspace/Users/vminh9560@gmail.com/recommender-service")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data_test" / "dwh_mock"
BPR_PATH = PROJECT_ROOT / "models" / "bpr" / "model.npz"
LOCAL_BGE_PATH = PROJECT_ROOT / "models" / "bge-reranker"
os.environ["BPR_CHECKPOINT_PATH"] = str(BPR_PATH)
os.environ["BGE_CHECKPOINT_PATH"] = str(LOCAL_BGE_PATH)

from src.config import settings
from src.db import LocalDataClient
from src.pipeline import (
    build_interaction_table,
    load_bpr_model,
    generate_candidates,
    build_user_profiles,
    build_item_descriptions,
    rerank_candidates,
)
from src.fallback import (
    filter_cold_books,
    get_user_interaction_counts,
    classify_users,
    get_popularity_recommendations,
    build_hybrid_recommendations,
)

# Prefer local BGE checkpoint if available
if LOCAL_BGE_PATH.exists():
    try:
        object.__setattr__(settings.reranker, "model_name", str(LOCAL_BGE_PATH))
    except Exception:
        pass

target_user_id = 1

with LocalDataClient(DATA_DIR) as client:
    interaction_df = build_interaction_table(client)

    if target_user_id is not None:
        interaction_df = interaction_df[interaction_df["user_id"] == target_user_id].copy()

    if interaction_df.empty:
        recs = get_popularity_recommendations(client, top_k=settings.reranker.top_k)
    else:
        interaction_df, _ = filter_cold_books(interaction_df)
        if interaction_df.empty:
            recs = get_popularity_recommendations(client, top_k=settings.reranker.top_k)
        else:
            bpr_model = load_bpr_model(str(BPR_PATH))
            candidates_df = generate_candidates(bpr_model, interaction_df)

            interaction_counts = get_user_interaction_counts(interaction_df)
            user_groups = classify_users([target_user_id], interaction_counts)

            if target_user_id in user_groups["warm"]:
                user_profiles = build_user_profiles(client, interaction_df)
                item_descriptions = build_item_descriptions(
                    client, candidates_df["candidate_book_id"].unique().tolist()
                )
                ranked = rerank_candidates(candidates_df, user_profiles, item_descriptions)
                recs = ranked.get(target_user_id, [])
            elif target_user_id in user_groups["hybrid"]:
                popularity_recs = get_popularity_recommendations(
                    client, top_k=settings.reranker.top_k
                )
                bpr_recs = [
                    {"book_id": int(r["candidate_book_id"]), "mf_score": float(r["mf_score"])}
                    for _, r in candidates_df.iterrows()
                ]
                recs = build_hybrid_recommendations(
                    bpr_results=bpr_recs,
                    popularity_results=popularity_recs,
                    top_k=settings.reranker.top_k,
                )
            else:
                recs = get_popularity_recommendations(client, top_k=settings.reranker.top_k)

recs